In [7]:
import os
import json
import time
from groq import Groq
import ollama
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
#CONFIGURATION

GROQ_MODEL = "llama-3.1-8b-instant"
OLLAMA_MODEL = "qwen3:8b"
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

#Track which backend is being used
current_backend = "groq"

def call_llm(system_prompt: str, user_prompt: str, max_tokens: int = 1000, temperature: float = 0.1) -> str:
    """
    This function is a unified call of the LLM with an automatic fallback:
    1. Try Groq (8b model)
    2. If Groq fails -> switch to local Ollama
    3. Retruns text output.
    """
    global current_backend

    if os.getenv("GROQ_API_KEY"):
        try:
            response = groq_client.chat.completions.create(
                model = GROQ_MODEL,
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens = max_tokens, #sets an upper limit on the number of tokens the model generates in response
                temperature = temperature, #controls randomness and probability distribution of the next token
                timeout = 30 #30-second timeout before giving up.
            )
            current_backeend = "groq"
            return response.choices[0].message.content
        except Exception as e:
            print(f"⚠ Groq unavailable ({type(e).__name__}). Switching to local Ollama...")
    
    #------------Fallback: Local Ollama-------------------
    try:
        response = ollama.chat(
            model = OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            options={"temperature": temperature, "num_predict": max_tokens}
        )
        current_backend="ollama"
        return response["message"]["content"]
    except Exception as e:
        raise RuntimeError(
            f"Both Groq and Ollama failed. Groq needs internet; "
            "Ollama needs to be running. Check both. \n"
            f"Ollama error: {e}"
        )

def get_active_backend() -> str:
    #Returns the current active backend - useful for Streamlit status display.
    return current_backend

def call_llm_with_retry(system_prompt, user_prompt, max_tokens = 1000, temperature = 0.1, retries = 3):
    #Full retry wrapper with backoff - to be used everywhere in the pipeline.
    for attempt in range(retries):
        try:
            return call_llm(system_prompt, user_prompt, max_tokens)
        except RuntimeError:
            raise
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise
        except RateLimitError:
            wait = 2 ** attempt
            print(f"Rate limit hit. Waiting {wait}s before retry {attempt+1}/{retries}...")
            time.sleep(wait)
            

In [15]:
call_llm_with_retry("Answer with justification", "Who is the father of Philisophy?")

⚠ Groq unavailable (APIConnectionError). Switching to local Ollama...


'The title "father of philosophy" is often attributed to **Socrates** in the context of **Western philosophy**. Here\'s the justification:\n\n1. **Socrates** (c. 470–399 BCE) is widely recognized as the foundational figure in Western philosophical thought. He is best known for his method of questioning, known as the **Socratic method**, which emphasized critical dialogue and the pursuit of truth through dialectical reasoning. His contributions laid the groundwork for ethical inquiry, logic, and epistemology, influencing later philosophers like Plato and Aristotle.\n\n2. **Thales of Miletus** (c. 624–546 BCE), often called the "first philosopher," is considered the **first philosopher** in the Western tradition. He sought natural explanations for phenomena (e.g., water as the origin of all things), moving away from mythological explanations. However, he is not typically labeled the "father of philosophy" in the same way as Socrates, as his work was more speculative and less systematic.\